Read [](file:///d%3A/Research/Under-water-metamaterial/Underwater-acoustics-absorption/references/paper_context.md#1-1), lines 1 to 100

Read [](file:///d%3A/Research/Under-water-metamaterial/Underwater-acoustics-absorption/notebooks/10_Advanced_Audit_and_Improved_Diffusion.ipynb#610-610), lines 610 to 793

Here is the complete architecture explanation with diagrams for the **Physics-Guided Conditional Diffusion Model (V3)** in this notebook.

---

## The Big Picture — Two Separate Networks

The model is made of **two distinct networks** that work together:



In [ ]:
┌─────────────────────────────────────────────────────────────────────┐
│  NETWORK 1: Forward Surrogate MLP  (trained first, then FROZEN)     │
│  Purpose: differentiable physics proxy                               │
│  20 params ──► [absorption scalar]                                  │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│  NETWORK 2: Noise Predictor (ConditionalDiffusionNetV3)              │
│  Purpose: learns to denoise parameter vectors                        │
│  [noisy params x_t] + [timestep t] + [target absorption c]          │
│                    ──► [predicted noise ε]                           │
└─────────────────────────────────────────────────────────────────────┘



---

## Network 1 — Forward Surrogate MLP

### Purpose
The TMM physics engine is not differentiable (it's a loop with numpy/PyTorch ops but you can't back-propagate through the whole training dataset efficiently). So you train this small MLP to **approximate TMM** in a smooth differentiable way. It is used **only during training** of Network 2 as a physics constraint.

### Architecture



In [ ]:
INPUT: 20 normalised parameters  (shape: [batch, 20])
         │
         ▼
   ┌─────────────┐
   │ Linear(20→256) │
   │ LeakyReLU(0.01)│
   │ BatchNorm1d    │
   └──────┬────────┘
          ▼
   ┌─────────────┐
   │ Linear(256→256)│
   │ LeakyReLU(0.01)│
   │ BatchNorm1d    │
   └──────┬────────┘
          ▼
   ┌─────────────┐
   │ Linear(256→128)│
   │ LeakyReLU(0.01)│
   └──────┬────────┘
          ▼
   ┌─────────────┐
   │ Linear(128→1) │
   │ Sigmoid       │  ← forces output to (0,1) = valid absorption range
   └──────┬────────┘
          ▼
OUTPUT: 1 scalar  [average absorption coefficient ∈ (0,1)]



**Why Sigmoid at the end?** Absorption is physically bounded between 0 and 1. Sigmoid enforces this hard constraint without any clipping.

**Training:** Adam, LR=1e-3, CosineAnnealing for 30 epochs, MSE loss against TMM-computed absorptions. Once trained, **all weights are frozen** (`requires_grad = False`).

---

## Network 2 — Noise Predictor (`ConditionalDiffusionNetV3`)

This is the **main model**. It is a function $\epsilon_\theta(x_t, t, c)$ that takes a noisy parameter vector, the timestep, and the target absorption, and predicts the noise that was added.

### Three Encoders — Three Inputs



In [ ]:
INPUT A: noisy params x_t          INPUT B: timestep t          INPUT C: absorption target c
shape [batch, 20]                  shape [batch, 1]              shape [batch, 1]
         │                                  │                              │
         ▼                                  ▼                              ▼
 ┌──────────────┐              ┌───────────────────────┐       ┌────────────────────┐
 │ input_proj   │              │ SinusoidalEmbeddings  │       │ cond_mlp           │
 │ Linear(20→512)│             │ → [batch, 512]        │       │ Linear(1→512)      │
 └──────┬───────┘              │ Linear(512→512) + SiLU│       │ SiLU               │
        │                      │ Linear(512→512)       │       │ Linear(512→512)    │
        │                      └──────────┬────────────┘       └────────┬───────────┘
        │                                 │                              │
        │                                 └─────────────┬───────────────┘
        │                                               │
        │                                  global_cond = t_emb + c_emb
        │                                          shape [batch, 512]
        │                                               │
        ▼                                               ▼
   h = [batch, 512]          ──────────────────► FiLMBlock × 4



---

### Sinusoidal Position Embeddings — Timestep Encoder



In [ ]:
Input: t  (scalar 0..99, shape [batch,1])
       │
       ▼
  half_dim = 512//2 = 256
  
  frequencies = exp( -log(10000) × [0,1,...,255] / 255 )
                ←── 256 exponentially spaced frequencies
  
  angles = t × frequencies        shape [batch, 256]
  
  output = [sin(angles), cos(angles)]   shape [batch, 512]
       │
       ▼
  Linear(512→512) + SiLU
  Linear(512→512)
       │
       ▼
  t_emb  [batch, 512]



**Why sinusoidal?** This is directly borrowed from transformer positional encoding. The network needs to know "how noisy" the input is. Different timesteps need very different denoising behaviours (step 99 = nearly pure noise, step 0 = nearly clean). Sinusoidal embedding gives a unique, smooth vector for every timestep integer.

**Why high frequency + low frequency together?** `sin` + `cos` at exponentially spaced frequencies means nearby timesteps have similar embeddings but are still distinguishable — the network can generalise between similar noise levels.

---

### Condition Encoder — Absorption Target



In [ ]:
Input: c  (target absorption, shape [batch, 1])
       │
       ▼
  Linear(1 → 512) + SiLU
  Linear(512 → 512)
       │
       ▼
  c_emb  [batch, 512]



**Why an MLP for a scalar?** The single scalar `c` needs to be projected into the same 512-dimensional space as the time embedding so they can be **added together**. The MLP learns a rich nonlinear representation of "what absorption level am I targeting."

---

### Global Conditioning Signal



In [ ]:
global_cond = t_emb + c_emb      shape [batch, 512]



**Why add them?** Both are projected to the same 512-d space and simply summed. This means the FiLM blocks receive a single **joint signal** encoding both *how noisy the input is* and *what absorption we want*. The network cannot achieve the right absorption without knowing the timestep, and vice versa.

---

### FiLM Block — The Core Building Block (×4)

**FiLM = Feature-wise Linear Modulation.** This is the central architectural innovation. It lets the condition (`global_cond`) **control how the hidden features are scaled and shifted**, layer by layer.



In [ ]:
                    ┌───────────────────────────────┐
RESIDUAL INPUT ──► ─┤                               ├─► OUTPUT
 h [batch,512]      │  ┌──────────────────────────┐ │
                    │  │ fc1: Linear(512→512)      │ │
                    │  │ LayerNorm(512)             │ │
                    │  │          │                 │ │
                    │  │     ┌────┴────┐            │ │
                    │  │     │FiLM     │◄── global_cond [batch,512]
                    │  │     │generator│    Linear(512 → 1024)
                    │  │     │         │    → split into [γ, β]
                    │  │     │         │    each shape [batch,512]
                    │  │     └────┬────┘            │ │
                    │  │          │                  │ │
                    │  │  h = h × (1 + γ) + β       │ │  ← FiLM modulation
                    │  │  SiLU activation            │ │
                    │  │  Dropout(0.1)               │ │
                    │  │  fc2: Linear(512→512)       │ │
                    │  └──────────────────────────┘  │
                    │  + residual (skip connection)   │
                    └───────────────────────────────┘



**What FiLM does, step by step:**
1. `fc1` projects features: `h → h'` (linear transform)
2. `LayerNorm` normalises features (zero mean, unit variance per sample)
3. `film_gen` takes `global_cond` and outputs two vectors: **γ (scale)** and **β (shift)**, each of size 512
4. `h' = h' × (1 + γ) + β` — the condition *literally rescales and re-centers every feature dimension*
5. `SiLU + Dropout + fc2` further processes
6. **Residual add** — adds back the original `h` before the block (prevents vanishing gradients, allows identity shortcut)

**Why FiLM over simple concatenation?** With concatenation, the condition is just appended and the network must learn to use it. With FiLM, the condition **directly modulates every feature**, giving much stronger control. It's analogous to how a key controls a lock at every pin simultaneously.

---

### Full Network Flow (ConditionalDiffusionNetV3)



In [ ]:
x_t [batch,20] ─► input_proj(Linear 20→512) ─► h [batch,512]
                                                        │
                                                   FiLMBlock 1
                                                   ┌────┴────┐
                                                   │  h      │◄── global_cond
                                                   └────┬────┘
                                                        │
                                                   FiLMBlock 2
                                                   ┌────┴────┐
                                                   │  h      │◄── global_cond
                                                   └────┬────┘
                                                        │
                                                   FiLMBlock 3
                                                   ┌────┴────┐
                                                   │  h      │◄── global_cond
                                                   └────┬────┘
                                                        │
                                                   FiLMBlock 4
                                                   ┌────┴────┐
                                                   │  h      │◄── global_cond
                                                   └────┬────┘
                                                        │
                                                   LayerNorm(512)
                                                   SiLU
                                                        │
                                            output_proj(Linear 512→20)
                                    (initialised to all zeros — starts predicting 0)
                                                        │
                                                        ▼
                                           ε_pred [batch, 20]   ← predicted noise



**Parameter count:** ~2.1 million trainable parameters.

**Why zero-init the output projection?** At step 0 of training, predicts zero noise. This means the first loss is purely from the noise distribution, not from random large predictions — gives more stable training start.

---

## The Diffusion Process — What Happens to Data

### Forward Process (Noising) — During Training Only



In [ ]:
Clean params x_0        Random noise ε ~ N(0,I)
[batch, 20]             [batch, 20]
        │                      │
        │    t randomly from {0,...,99}
        │                      │
        ▼                      ▼
x_t = √(ᾱ_t) × x_0  +  √(1-ᾱ_t) × ε



$\bar{\alpha}_t = \prod_{s=1}^{t}(1 - \beta_s)$ — computed from the cosine beta schedule.

- At **t=0**: $\bar{\alpha}_0 \approx 1$ → $x_t \approx x_0$ (nearly clean)
- At **t=99**: $\bar{\alpha}_{99} \approx 0$ → $x_t \approx \epsilon$ (nearly pure noise)

### Cosine Beta Schedule



In [ ]:
t:    0      10      20     ...    50    ...    99
β_t:  tiny  small  small  ...  medium  ...  large

ᾱ_t:  ~1.0   ~0.9   ~0.8  ...   ~0.5  ...   ~0.0
       │                                        │
       clean                                 pure noise



**Why cosine instead of linear?** Linear schedules add noise too fast at the start and too slowly at the end. The cosine schedule is symmetric and smooth — the model spends balanced amounts of training time at each noise level.

---

## The Dual Loss — The Key Contribution

This is what makes this "physics-guided." There are **two losses** added together every training step:



In [ ]:
                         ┌──────────────────────────────────────────┐
                         │         ONE TRAINING STEP                │
                         │                                          │
  x_0 [batch,20] ──────► Noising ──► x_t ──► Network ──► ε_pred   │
                               ↑                                    │
                               ε (true noise)                       │
                                                                    │
  ┌─────────────────────────────────────────────────────────────┐   │
  │  LOSS 1 — Noise MSE (standard DDPM loss)                   │   │
  │  L_noise = MSE(ε_pred,  ε)                                  │   │
  │  Trains the network to correctly predict what noise was     │   │
  │  added. This is the core diffusion objective.               │   │
  └─────────────────────────────────────────────────────────────┘   │
                                                                    │
  ┌─────────────────────────────────────────────────────────────┐   │
  │  LOSS 2 — Physics MSE (in-the-loop absorption constraint)  │   │
  │                                                             │   │
  │  Step A: Reconstruct x_0 from prediction:                  │   │
  │    x̂_0 = (x_t - √(1-ᾱ_t)·ε_pred) / √(ᾱ_t)              │   │
  │                                                             │   │
  │  Step B: Clamp to [0,1] (normalised parameter space)       │   │
  │    x̂_0_clamped = clamp(x̂_0, 0, 1)                        │   │
  │                                                             │   │
  │  Step C: Run through FROZEN surrogate MLP                  │   │
  │    α̂ = Surrogate(x̂_0_clamped)    shape [batch,1]         │   │
  │                                                             │   │
  │  Step D: Compare against target absorption c               │   │
  │    L_phys = MSE(α̂,  c)                                    │   │
  │                                                             │   │
  │  If the predicted parameters don't produce the right       │   │
  │  absorption, this loss penalises the network.              │   │
  └─────────────────────────────────────────────────────────────┘   │
                                                                    │
  TOTAL LOSS = L_noise  +  λ · L_phys                              │
                                                                    │
  λ ramps from 0 → 0.1 over first 15 epochs (physics warmup)       │
  (avoids destabilising training before noise loss is learned)      │
                                                                    │
  → optimizer.zero_grad()                                           │
  → loss.backward()          (gradients flow through both losses)   │
  → clip_grad_norm(max=1.0)  (prevents exploding gradients)        │
  → optimizer.step()                                                │
  → EMA update                                                      │
  └──────────────────────────────────────────────────────────────────┘



**Why physics warmup (λ ramps from 0)?** At the beginning of training the network is random — its first predictions of `x̂_0` are garbage, so the physics loss would produce huge misleading gradients. Wait for the noise loss to first teach the network the basic denoising behavior, then gradually introduce the physics constraint on top.

---

## EMA — Exponential Moving Average

A **shadow copy** of the network weights is maintained:



In [ ]:
After every training step:

  EMA_weights ← 0.999 × EMA_weights  +  0.001 × current_weights

At INFERENCE (sampling):  use EMA weights, not current weights



**Why?** Training has noisy gradient updates — the weights jump around. EMA is the "running average" of where the weights have been, producing a smoother, more stable model. With decay=0.999, the EMA responds slowly to changes — it effectively averages over the last ~1000 training steps.

---

## Reverse Sampling — How You Generate Parameters

At inference time you go **backwards through the diffusion process** (100 → 0):



In [ ]:
Step 99:  x_99 ~ N(0, I)    ← start from pure Gaussian noise
          │
          ▼
   ε_pred = EMA_net(x_99, t=99, c)       ← predict the noise
   mean = (1/√α_99) × (x_99 - (1-α_99)/√(1-ᾱ_99) × ε_pred)
   x_98 = mean + √β_99 × z,    z~N(0,I) ← add small noise back
          │
          ▼  (repeat 99 times...)
          │
   Every 10 steps:
     ┌──────────────────────────────────────────┐
     │  CONSTRAINT ENFORCEMENT                  │
     │  x → inverse_scaler → clip to bounds → scaler
     │  Ensures parameters stay physically valid │
     └──────────────────────────────────────────┘
          │
          ▼
Step  0:  x_0   ← no noise added at final step
          │
          ▼
   inverse_scaler(x_0) → 20 physical parameters (mm, kg/m³, Pa...)



---

## Complete System Diagram



In [ ]:
═══════════════════════════════════════════════════════════════════
                        TRAINING PHASE
═══════════════════════════════════════════════════════════════════

 Database          x_0 [batch, 20]        cond c [batch, 1]
 100k samples ──►  normalised params  +   target absorption
                          │                       │
                          ▼                       │
                 ──────────────────               │
                 FORWARD DIFFUSION                │
                 t ~ Uniform(0,99)                │
                 ε ~ N(0, I)                      │
                 x_t = √ᾱ_t·x_0 + √(1-ᾱ_t)·ε   │
                 ──────────────────               │
                          │                       │
                          ▼                       ▼
                ┌─────────────────────────────────────┐
                │   ConditionalDiffusionNetV3          │
                │                                     │
                │   [t] ──► SinEmbedding ──► t_emb   │
                │   [c] ──► cond_mlp ──────► c_emb   │
                │           t_emb + c_emb = global_cond│
                │                                     │
                │   [x_t] ──► input_proj              │
                │               ▼                     │
                │           FiLMBlock 1 ◄─ global_cond│
                │               ▼                     │
                │           FiLMBlock 2 ◄─ global_cond│
                │               ▼                     │
                │           FiLMBlock 3 ◄─ global_cond│
                │               ▼                     │
                │           FiLMBlock 4 ◄─ global_cond│
                │               ▼                     │
                │           LayerNorm + SiLU           │
                │               ▼                     │
                │           output_proj               │
                └──────────────┬──────────────────────┘
                               │
                          ε_pred [batch, 20]
                               │
              ┌────────────────┴──────────────────────┐
              │                                       │
              ▼                                       ▼
   L_noise = MSE(ε_pred, ε)          Reconstruct x̂_0 from ε_pred
                                                      │
                                            Frozen Surrogate MLP
                                            20 params → absorption
                                                      │
                                      L_phys = MSE(ŷ_absorption, c)
              │                                       │
              └────────────┬──────────────────────────┘
                           │
              L_total = L_noise + λ·L_phys
                           │
              ┌────────────▼────────────┐
              │  Backprop + Grad Clip   │
              │  Adam + CosineAnneal LR │
              │  EMA weight update      │
              └─────────────────────────┘

═══════════════════════════════════════════════════════════════════
                        INFERENCE PHASE
═══════════════════════════════════════════════════════════════════

  User inputs target absorption c (e.g. 0.325678)
                    │
                    ▼
          x_T ~ N(0, I)    [batch, 20]
                    │
          for t = 99 down to 0:
              │
              ▼
          EMA_net(x_t, t, c) ──► ε_pred
              │
          x_{t-1} = DDPM reverse step
              │
          every 10 steps: clip to physical bounds
              │
          t = t - 1
              │
          (repeat)
                    │
                    ▼
          x_0 → inverse_scaler → 20 physical parameters
          (d1..d10, m2,m3,m5,m6,m8,m9, ρ, η, E, ν)
                    │
                    ▼
          TMM physics verification (GPU)
                    │
                    ▼
          Best candidate selected from 20 generated



---

## Key Design Decisions Summary

| Decision | What | Why |
|---|---|---|
| **FiLM modulation** | Condition controls scale+shift of every feature | Stronger conditioning than concatenation |
| **Sinusoidal time embed** | Unique smooth vector per timestep | Network can interpolate between noise levels |
| **Cosine β schedule** | Smooth symmetric noise addition | Better distribution of training across noise levels vs linear |
| **Dual loss** | Noise MSE + Physics MSE | Forces generated params to actually produce correct absorption |
| **Frozen surrogate** | No gradients flow into surrogate | Surrogate is a fixed physics proxy, not jointly optimized |
| **Physics warmup** | λ: 0 → 0.1 over 15 epochs | Prevents physics loss from destabilizing early training |
| **EMA** | Shadow weights with 0.999 decay | Smooth stable weights for inference |
| **Grad clipping = 1.0** | Clip gradient norm | Prevents exploding gradients from physics loss |
| **Zero-init output** | Output projection starts at zeros | Stable training initialization |
| **Constraint enforcement** | Clip to bounds every 10 steps during sampling | Keeps intermediate samples in valid physical space |